# Supplementary Multidimensional Similarity Analysis (Colab)

This version fixes the all-NaN issue by using a **layered ad-vector strategy**:

1. Try ad-signal columns (`OPE_cond`, `CON_cond`, `EXT_cond`, `AGR_cond`, `NEU_cond`) directly.
2. If signals are sparse, fill missing signal cells with `0` before ad-level means.
3. If still unusable, fallback to participant trait means by ad condition.

Outputs:
- Table 7: Ad personality profile matrix (5D)
- Table 8: Similarity metrics comparison
- Table 9: Cross-group validation
- Regression and appendix figures


In [ ]:
import io
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
import statsmodels.api as sm

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


## 1) Upload data


In [ ]:
from google.colab import files
uploaded = files.upload()

filename = list(uploaded.keys())[0]
print("Loaded:", filename)

if filename.lower().endswith((".xls", ".xlsx")):
    df = pd.read_excel(io.BytesIO(uploaded[filename]))
else:
    try:
        df = pd.read_csv(io.BytesIO(uploaded[filename]), low_memory=False)
    except Exception:
        df = pd.read_csv(io.BytesIO(uploaded[filename]), sep="	", low_memory=False)

print("Shape:", df.shape)
df.head(5)


## 2) Clean and map variables


In [ ]:
null_tokens = ["#NULL!", "", " ", "NA", "N/A", "nan", "NaN", "None"]
df = df.replace(null_tokens, np.nan)

trait_raw = ["OPEN_Score", "CONSC_Score", "EXTRA_Score", "AGREE_Score", "NEURO_Score"]
trait_z = ["ZOPEN_Score", "ZCONSC_Score", "ZEXTRA_Score", "ZAGREE_Score", "ZNEURO_Score"]
ad_signal_cols = ["OPE_cond", "CON_cond", "EXT_cond", "AGR_cond", "NEU_cond"]

click_items = ["CLIC1", "CLIC2", "CLIC3", "CLIC4"]
pers_items = ["PERS1", "PERS2", "PERS3", "PERS4", "PERS5", "PERS6"]
rele_items = ["RELE1", "RELE2", "RELE3"]

for c in set(trait_raw + trait_z + ad_signal_cols + click_items + pers_items + rele_items):
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

if any(c in df.columns for c in click_items):
    df["CLICK_MEAN"] = df[[c for c in click_items if c in df.columns]].mean(axis=1)
if any(c in df.columns for c in pers_items):
    df["PERSUASIVE_MEAN"] = df[[c for c in pers_items if c in df.columns]].mean(axis=1)
if any(c in df.columns for c in rele_items):
    df["RELEVANCE_MEAN"] = df[[c for c in rele_items if c in df.columns]].mean(axis=1)

print("AdCond present:", "AdCond" in df.columns)


## 3) Diagnose repeated-measures and missingness


In [ ]:
id_col = "ResponseId" if "ResponseId" in df.columns else None
if id_col and "AdCond" in df.columns:
    print("Ad conditions per respondent:")
    print(df.groupby(id_col)["AdCond"].nunique().describe())

avail_signals = [c for c in ad_signal_cols if c in df.columns]
if avail_signals:
    miss = df[avail_signals].isna().mean().sort_values(ascending=False)
    print("\nMissing share by ad-signal column:")
    display(miss.to_frame("missing_share"))


## 4) Build Table 7 ad vectors (robust, no-all-NaN fallback)


In [ ]:
if "AdCond" not in df.columns:
    raise ValueError("AdCond column is required.")

rename_map = {
    "OPE_cond": "Openness",
    "CON_cond": "Conscientiousness",
    "EXT_cond": "Extraversion",
    "AGR_cond": "Agreeableness",
    "NEU_cond": "Neuroticism",
}
vector_order = ["Openness", "Conscientiousness", "Extraversion", "Agreeableness", "Neuroticism"]

ad_vector = None
vector_source = None

avail_signals = [c for c in ad_signal_cols if c in df.columns]
if len(avail_signals) == 5:
    # Attempt 1: direct means over observed values
    temp = df.dropna(subset=["AdCond"]).groupby("AdCond", dropna=True)[avail_signals].mean().rename(columns=rename_map)

    # If too sparse, attempt 2: fill missing signal cells with 0 before grouping
    sparse_problem = temp[vector_order].isna().mean().mean() > 0.40
    if sparse_problem:
        temp2 = (
            df.dropna(subset=["AdCond"])
              .assign(**{c: df[c].fillna(0) for c in avail_signals})
              .groupby("AdCond", dropna=True)[avail_signals]
              .mean()
              .rename(columns=rename_map)
        )
        temp = temp2
        vector_source = "ad_signal_columns_fillna0"
    else:
        vector_source = "ad_signal_columns_observed"

    # Keep only required order
    ad_vector = temp.reindex(columns=vector_order)

# Final fallback: participant trait means
if ad_vector is None or ad_vector[vector_order].isna().all().all():
    if set(trait_raw).issubset(df.columns):
        ad_vector = (
            df.dropna(subset=["AdCond"] + trait_raw)
              .groupby("AdCond", dropna=True)[trait_raw]
              .mean()
              .rename(columns={
                  "OPEN_Score": "Openness",
                  "CONSC_Score": "Conscientiousness",
                  "EXTRA_Score": "Extraversion",
                  "AGREE_Score": "Agreeableness",
                  "NEURO_Score": "Neuroticism",
              })
              .reindex(columns=vector_order)
        )
        vector_source = "fallback_trait_means"
    else:
        raise ValueError("Could not build ad vectors. Need either 5 ad-signal columns or 5 trait score columns.")

# Impute any remaining cell-level NaN by column mean, then 0 (prevents downstream all-NaN tables)
ad_vector = ad_vector.copy()
ad_vector = ad_vector.apply(lambda col: col.fillna(col.mean()), axis=0).fillna(0)

print("Ad vector source:", vector_source)
print("Table 7")
table7 = ad_vector.round(3)
display(table7)


## 5) Participant-to-ad similarity


In [ ]:
# Prefer z-score participant traits only if they have usable non-null content
use_z = set(trait_z).issubset(df.columns) and (df[trait_z].notna().mean().mean() > 0.20)

if use_z:
    p_cols = trait_z
    p_map = {
        "ZOPEN_Score": "Openness",
        "ZCONSC_Score": "Conscientiousness",
        "ZEXTRA_Score": "Extraversion",
        "ZAGREE_Score": "Agreeableness",
        "ZNEURO_Score": "Neuroticism",
    }
elif set(trait_raw).issubset(df.columns):
    p_cols = trait_raw
    p_map = {
        "OPEN_Score": "Openness",
        "CONSC_Score": "Conscientiousness",
        "EXTRA_Score": "Extraversion",
        "AGREE_Score": "Agreeableness",
        "NEURO_Score": "Neuroticism",
    }
else:
    raise ValueError("Participant trait columns not found.")

analysis_df = df.dropna(subset=["AdCond"] + p_cols).copy()
for c in p_cols:
    analysis_df[c] = pd.to_numeric(analysis_df[c], errors="coerce")

# fill rare participant missing values to avoid row-wise NaN similarities
analysis_df[p_cols] = analysis_df[p_cols].apply(lambda s: s.fillna(s.mean())).fillna(0)

ad_vectors_np = ad_vector.to_dict("index")


def euclidean(a, b):
    return float(np.linalg.norm(np.array(a) - np.array(b)))


def cosine(a, b):
    a = np.array(a).reshape(1, -1)
    b = np.array(b).reshape(1, -1)
    return float(cosine_similarity(a, b)[0, 0])


def row_similarity(row):
    ad = row["AdCond"]
    if ad not in ad_vectors_np:
        return pd.Series({"euclidean": np.nan, "cosine": np.nan})
    p_named = {p_map[c]: row[c] for c in p_cols}
    p_vec = [p_named[k] for k in vector_order]
    a_vec = [ad_vectors_np[ad][k] for k in vector_order]
    return pd.Series({"euclidean": euclidean(p_vec, a_vec), "cosine": cosine(p_vec, a_vec)})

analysis_df[["euclidean", "cosine"]] = analysis_df.apply(row_similarity, axis=1)
print("Similarity rows with non-null cosine:", analysis_df["cosine"].notna().sum())
analysis_df[["AdCond", "euclidean", "cosine"]].head()


## 6) Table 8 similarity comparison


In [ ]:
def safe_corr(x, y):
    tmp = pd.DataFrame({"x": x, "y": y}).dropna()
    if len(tmp) < 3:
        return np.nan
    return tmp["x"].corr(tmp["y"])

work = analysis_df.dropna(subset=["euclidean", "cosine"]).copy()

table8 = work.groupby("AdCond").agg(
    Euclidean_Distance_Mean=("euclidean", "mean"),
    Cosine_Similarity_Mean=("cosine", "mean"),
    Corr_Cosine_Click=("cosine", lambda s: safe_corr(s, work.loc[s.index, "CLICK_MEAN"]) if "CLICK_MEAN" in work.columns else np.nan),
    Corr_Euclidean_Click=("euclidean", lambda s: safe_corr(s, work.loc[s.index, "CLICK_MEAN"]) if "CLICK_MEAN" in work.columns else np.nan),
).round(3)

print("Table 8")
display(table8)


## 7) Table 9 cross-group validation


In [ ]:
work = analysis_df.dropna(subset=["cosine"]).copy()

if "connectId" in work.columns:
    cid_num = pd.to_numeric(work["connectId"], errors="coerce")
    if cid_num.notna().sum() > 0:
        cutoff = cid_num.median()
        work["grp"] = np.where(cid_num <= cutoff, "Group 1", "Group 2")
    else:
        work["grp"] = np.where(work["connectId"].astype(str).str.len() % 2 == 0, "Group 1", "Group 2")
elif "ResponseId" in work.columns:
    work["grp"] = np.where(work["ResponseId"].astype(str).str.len() % 2 == 0, "Group 1", "Group 2")
else:
    rng = np.random.default_rng(42)
    work["grp"] = np.where(rng.random(len(work)) < 0.5, "Group 1", "Group 2")

pivot = work.pivot_table(index="AdCond", columns="grp", values="cosine", aggfunc="mean")

if {"Group 1", "Group 2"}.issubset(pivot.columns):
    r_groups = pivot[["Group 1", "Group 2"]].corr().iloc[0, 1]
else:
    r_groups = np.nan

table9 = pivot.rename(columns={"Group 1": "Cosine_Group1", "Group 2": "Cosine_Group2"}).round(3)
table9["r_between_groups"] = np.round(r_groups, 3) if pd.notna(r_groups) else np.nan

print("Table 9")
display(table9)


## 8) Regression


In [ ]:
if "CLICK_MEAN" in analysis_df.columns:
    reg = analysis_df[["CLICK_MEAN", "euclidean", "cosine"]].dropna().copy()
    if len(reg) >= 5:
        m1 = sm.OLS(reg["CLICK_MEAN"], sm.add_constant(reg[["euclidean"]])).fit()
        m2 = sm.OLS(reg["CLICK_MEAN"], sm.add_constant(reg[["cosine"]])).fit()
        print("Model A: CLICK_MEAN ~ euclidean")
        print(m1.summary())
        print("\nModel B: CLICK_MEAN ~ cosine")
        print(m2.summary())
    else:
        print("Not enough rows for regression after filtering.")
else:
    print("CLICK_MEAN not available.")


## 9) Appendix figures


In [ ]:
# 9.1 PCA 3D
if len(ad_vector) >= 3:
    pca = PCA(n_components=3)
    vec_3d = pca.fit_transform(ad_vector[vector_order].values)
    fig = plt.figure(figsize=(8,6))
    ax = fig.add_subplot(111, projection='3d')
    ax.scatter(vec_3d[:,0], vec_3d[:,1], vec_3d[:,2], s=80)
    for i, name in enumerate(ad_vector.index):
        ax.text(vec_3d[i,0], vec_3d[i,1], vec_3d[i,2], name)
    ax.set_title("3D Projection of Advertisement 5D Trait Vectors (PCA)")
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.set_zlabel("PC3")
    plt.show()
else:
    print("Not enough ad conditions for 3D PCA plot.")

# 9.2 Radar
labels = ["Open", "Consc", "Extra", "Agree", "Neuro"]
angles = np.linspace(0, 2*np.pi, len(labels), endpoint=False)
angles = np.concatenate([angles, [angles[0]]])

fig, ax = plt.subplots(figsize=(8,8), subplot_kw={"polar": True})
for ad, row in ad_vector.iterrows():
    vals = [row[k] for k in vector_order]
    vals = np.concatenate([vals, [vals[0]]])
    ax.plot(angles, vals, linewidth=2, label=ad)
    ax.fill(angles, vals, alpha=0.05)
ax.set_xticks(angles[:-1]); ax.set_xticklabels(labels)
ax.set_title("Radar Chart: Multidimensional Spillover Across Ad Conditions")
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.10))
plt.show()

# 9.3 Cosine angle
example = analysis_df.dropna(subset=["AdCond", "cosine"]).iloc[0]
ad_name = example["AdCond"]
p_named = {p_map[c]: example[c] for c in p_cols}
p_vec = np.array([p_named[k] for k in vector_order], dtype=float)
a_vec = np.array([ad_vectors_np[ad_name][k] for k in vector_order], dtype=float)
cos_val = cosine_similarity(p_vec.reshape(1,-1), a_vec.reshape(1,-1))[0,0]
angle_deg = np.degrees(np.arccos(np.clip(cos_val, -1, 1)))

plt.figure(figsize=(6,4))
plt.bar(["Cosine Similarity"], [cos_val])
plt.ylim(-1, 1)
plt.title(f"Cosine Alignment for Example Case ({ad_name})\nAngle = {angle_deg:.1f}°")
plt.show()
print(f"Cosine similarity: {cos_val:.3f}")
print(f"Angle (degrees): {angle_deg:.2f}")


## 10) Export CSVs


In [ ]:
out_dir = "outputs"
os.makedirs(out_dir, exist_ok=True)

table7.to_csv(f"{out_dir}/table7_ad_profile_matrix.csv")
table8.to_csv(f"{out_dir}/table8_similarity_comparison.csv")
table9.to_csv(f"{out_dir}/table9_cross_group_validation.csv")

print("Saved files:")
for f in sorted(os.listdir(out_dir)):
    print("-", os.path.join(out_dir, f))

from google.colab import files
for f in ["table7_ad_profile_matrix.csv", "table8_similarity_comparison.csv", "table9_cross_group_validation.csv"]:
    files.download(os.path.join(out_dir, f))
